# From Scratch: Synthetic MCQs + TinyLlama/Phi-2 LoRA + OCR Inference

This notebook builds the full pipeline from scratch:

1. Generate about 3000 synthetic deep-learning MCQ examples and PNG images.
2. Train TinyLlama/Phi-2 with LoRA on synthetic text labels.
3. Show validation accuracy on held-out synthetic MCQs.
4. OCR test PNG images and predict options.
5. Write `submission.csv`.

Important: hidden-test accuracy is computed by the evaluator's grading script. You can only show accuracy on synthetic validation data because hidden labels are not available.

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Environment ready")

In [ ]:
import csv
import gc
import json
import random
import re
import textwrap
from dataclasses import dataclass
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

try:
    import pytesseract
except Exception as exc:
    pytesseract = None
    print("pytesseract unavailable:", exc)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Config

In [ ]:
PROJECT_DIR = Path.cwd()
SYNTHETIC_DIR = PROJECT_DIR / "synthetic_train"
SYNTHETIC_IMAGE_DIR = SYNTHETIC_DIR / "images"
METADATA_PATH = SYNTHETIC_DIR / "metadata.jsonl"
TRAIN_CSV = SYNTHETIC_DIR / "train.csv"

TEST_CSV = PROJECT_DIR / "test.csv"
IMAGE_DIR = PROJECT_DIR / "images"
SAMPLE_SUBMISSION = PROJECT_DIR / "sample_submission.csv"
SUBMISSION_PATH = PROJECT_DIR / "submission.csv"

NUM_SYNTHETIC_QUESTIONS = 3000
MAX_TRAIN_SAMPLES = 2800
MAX_EVAL_SAMPLES = 200
MODEL_CHOICE = "tinyllama"  # change to "phi2" only if phi-2 is downloaded

MODEL_CANDIDATES = {
    "tinyllama": [
        PROJECT_DIR / "TinyLlama-1.1B-Chat-v1.0",
        Path("/home/aditig/wheels/TinyLlama-1.1B-Chat-v1.0"),
        Path("/scratch/karthik_NEW/aditig/dl_project/TinyLlama-1.1B-Chat-v1.0"),
    ],
    "phi2": [
        PROJECT_DIR / "phi-2",
        Path("/home/aditig/wheels/phi-2"),
        Path("/scratch/karthik_NEW/aditig/dl_project/phi-2"),
    ],
}
BASE_MODEL_DIR = next((p for p in MODEL_CANDIDATES[MODEL_CHOICE] if p.exists()), None)
if BASE_MODEL_DIR is None:
    raise FileNotFoundError("Download TinyLlama/Phi-2 first. Checked: " + ", ".join(map(str, MODEL_CANDIDATES[MODEL_CHOICE])))

ADAPTER_DIR = PROJECT_DIR / ("tinyllama_mcq_lora" if MODEL_CHOICE == "tinyllama" else "phi2_mcq_lora")
SEED = 42
MAX_LENGTH = 512
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

random.seed(SEED)
torch.manual_seed(SEED)
print("PROJECT_DIR:", PROJECT_DIR)
print("BASE_MODEL_DIR:", BASE_MODEL_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)

## Generate Synthetic MCQ Text and PNG Images

In [ ]:
@dataclass(frozen=True)
class MCQ:
    question: str
    answer: str
    distractors: tuple
    topic: str


QUESTION_BANK = [
    MCQ("Which layer type is most commonly used to extract local spatial features from images?", "Convolutional layer", ("Dropout layer", "Softmax layer", "Embedding lookup"), "CNN"),
    MCQ("What does max pooling usually do in a CNN?", "Reduces spatial resolution while keeping strong activations", ("Increases the number of classes", "Initializes all kernels", "Computes the loss"), "CNN"),
    MCQ("Why are recurrent neural networks used for sequence data?", "They maintain hidden state across time steps", ("They remove all temporal order", "They require no parameters", "They only process images"), "RNN"),
    MCQ("Which problem do LSTM gates help reduce?", "Vanishing gradients in long sequences", ("Class imbalance", "Image aliasing", "One-hot encoding"), "RNN"),
    MCQ("What is the main role of self-attention in a Transformer?", "It lets tokens attend to other tokens in the sequence", ("It crops input images", "It freezes all weights", "It performs data augmentation"), "Transformers"),
    MCQ("What gives Transformers information about token order?", "Positional encoding", ("Dropout mask", "Batch size", "Weight decay only"), "Transformers"),
    MCQ("In a GAN, what is the generator trained to do?", "Produce samples that fool the discriminator", ("Assign class labels only", "Normalize gradients", "Delete noisy samples"), "GAN"),
    MCQ("In a GAN, what does the discriminator learn?", "Distinguish real samples from generated samples", ("Compute BLEU score", "Choose optimizer type", "Store the dataset"), "GAN"),
    MCQ("What is the latent variable in a VAE used for?", "Represent compressed factors of variation", ("Store file names", "Replace the optimizer", "Force labels to be integers"), "VAE"),
    MCQ("Which term in a VAE encourages the approximate posterior to match a prior?", "KL divergence", ("Cross-validation", "Max pooling", "Argmax"), "VAE"),
    MCQ("What is the purpose of gradient descent?", "Update parameters to reduce loss", ("Increase image size", "Encode labels as strings", "Remove validation data"), "Optimization"),
    MCQ("Which optimizer uses adaptive first and second moment estimates?", "Adam", ("ReLU", "Dropout", "Softmax"), "Optimization"),
    MCQ("Which loss is normally used for single-label multi-class classification?", "Cross-entropy loss", ("Mean absolute percentage error", "IoU only", "Silhouette score"), "Loss functions"),
    MCQ("What does mean squared error measure?", "Average squared difference between prediction and target", ("Number of classes", "GPU memory", "Vocabulary length"), "Loss functions"),
    MCQ("In PyTorch, which method switches a model to evaluation mode?", "model.eval()", ("model.train(FalseLabels)", "torch.freeze()", "optimizer.zero_eval()"), "PyTorch"),
    MCQ("In PyTorch training, why call optimizer.zero_grad()?", "To clear accumulated gradients before backpropagation", ("To delete model weights", "To load images", "To change labels"), "PyTorch"),
    MCQ("What does backpropagation compute?", "Gradients of the loss with respect to parameters", ("Final test accuracy directly", "Number of hidden units", "Image resolution"), "Backpropagation"),
    MCQ("Which rule is central to backpropagation?", "Chain rule", ("Bayes rule only", "Triangle inequality", "Majority vote"), "Backpropagation"),
    MCQ("What is a common purpose of dropout?", "Reduce overfitting by randomly disabling units during training", ("Increase overfitting deliberately", "Convert logits to labels", "Make inference slower"), "Regularization"),
    MCQ("What does weight decay usually penalize?", "Large parameter values", ("Small batch sizes", "Correct labels", "Input filenames"), "Regularization"),
]

QUESTION_VARIANTS = ["{q}", "Choose the correct answer: {q}", "Question: {q}", "In deep learning, {q}"]
WIDTH, HEIGHT = 1700, 2200
MARGIN_X, TOP_Y = 120, 120


def get_font(size):
    for name in ["DejaVuSans.ttf", "Arial.ttf", "LiberationSans-Regular.ttf"]:
        try:
            return ImageFont.truetype(name, size)
        except Exception:
            pass
    return ImageFont.load_default()


def draw_wrapped(draw, xy, text, font, fill=(20, 20, 20), width=62, line_gap=12):
    x, y = xy
    for line in textwrap.wrap(text, width=width):
        draw.text((x, y), line, font=font, fill=fill)
        y += font.size + line_gap
    return y


def make_synthetic_record(idx):
    item = random.choice(QUESTION_BANK)
    question = random.choice(QUESTION_VARIANTS).format(q=item.question)
    options = [item.answer] + list(item.distractors)
    random.shuffle(options)
    correct_option = options.index(item.answer) + 1
    image_name = f"synthetic_{idx:05d}"
    return {
        "image_name": image_name,
        "file": f"images/{image_name}.png",
        "question": question,
        "options": options,
        "correct_option": correct_option,
        "topic": item.topic,
    }


def render_record(record, out_path):
    image = Image.new("RGB", (WIDTH, HEIGHT), "white")
    draw = ImageDraw.Draw(image)
    title_font = get_font(54)
    body_font = get_font(46)
    small_font = get_font(34)
    y = TOP_Y
    draw.text((MARGIN_X, y), "Deep Learning MCQ", font=title_font, fill=(0, 0, 0))
    y += 100
    y = draw_wrapped(draw, (MARGIN_X, y), record["question"], body_font, width=52)
    y += 50
    for i, opt in enumerate(record["options"], start=1):
        y = draw_wrapped(draw, (MARGIN_X + 20, y), f"{i}. {opt}", body_font, width=56)
        y += 34
    draw.text((MARGIN_X, HEIGHT - 120), f"Topic: {record['topic']}", font=small_font, fill=(90, 90, 90))
    image.save(out_path)


def generate_synthetic_dataset(num_questions=3000, force=False):
    SYNTHETIC_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
    if METADATA_PATH.exists() and not force:
        existing = sum(1 for _ in METADATA_PATH.open("r", encoding="utf-8"))
        if existing >= num_questions:
            print("Using existing synthetic dataset:", METADATA_PATH)
            return

    with METADATA_PATH.open("w", encoding="utf-8") as meta, TRAIN_CSV.open("w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=["image_name", "correct_option", "topic"])
        writer.writeheader()
        for idx in range(1, num_questions + 1):
            record = make_synthetic_record(idx)
            render_record(record, SYNTHETIC_DIR / record["file"])
            meta.write(json.dumps(record) + "\n")
            writer.writerow({"image_name": record["image_name"], "correct_option": record["correct_option"], "topic": record["topic"]})
            if idx % 500 == 0:
                print("generated", idx)


generate_synthetic_dataset(NUM_SYNTHETIC_QUESTIONS, force=False)

## Load Synthetic Records and Build Prompts

In [ ]:
def load_records(path):
    records = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    random.shuffle(records)
    return records


SYSTEM_INSTRUCTION = "You answer deep learning multiple-choice questions. Return only one number: 1, 2, 3, 4, or 5."


def format_mcq_text(question, options):
    lines = [str(question).strip(), ""]
    for i, option in enumerate(options, start=1):
        lines.append(f"{i}. {str(option).strip()}")
    return "\n".join(lines)


def build_prompt(mcq_text):
    return f"### Instruction:\n{SYSTEM_INSTRUCTION}\n\n### Question:\n{mcq_text}\n\n### Answer:\n"


records = load_records(METADATA_PATH)
records = records[: MAX_TRAIN_SAMPLES + MAX_EVAL_SAMPLES]
eval_records = records[:MAX_EVAL_SAMPLES]
train_records = records[MAX_EVAL_SAMPLES:]
print("train:", len(train_records), "eval:", len(eval_records))
print(train_records[0])

## Load TinyLlama/Phi-2 and Add LoRA

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=DTYPE,
    local_files_only=True,
    trust_remote_code=True,
).to(DEVICE)
model.config.use_cache = False
model.gradient_checkpointing_enable()

def pick_lora_targets(model):
    wanted = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "dense", "fc1", "fc2"}
    found = set()
    for name, module in model.named_modules():
        if isinstance(module, torch.nn.Linear):
            suffix = name.split(".")[-1]
            if suffix in wanted and suffix != "lm_head":
                found.add(suffix)
    return sorted(found) or ["q_proj", "v_proj"]

target_modules = pick_lora_targets(model)
print("LoRA targets:", target_modules)
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=target_modules)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Dataset and Training

In [ ]:
class MCQTextDataset(Dataset):
    def __init__(self, records, tokenizer, max_length):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        mcq_text = format_mcq_text(r["question"], r["options"])
        prompt = build_prompt(mcq_text)
        answer = str(int(r["correct_option"])) + self.tokenizer.eos_token
        full = prompt + answer
        enc = self.tokenizer(full, truncation=True, max_length=self.max_length, padding="max_length", return_tensors="pt")
        prompt_enc = self.tokenizer(prompt, truncation=True, max_length=self.max_length, return_tensors="pt")
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        labels = input_ids.clone()
        prompt_len = min(prompt_enc["input_ids"].shape[1], labels.shape[0])
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


train_dataset = MCQTextDataset(train_records, tokenizer, MAX_LENGTH)
eval_dataset = MCQTextDataset(eval_records, tokenizer, MAX_LENGTH)

training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=20,
    save_steps=300,
    eval_steps=300,
    eval_strategy="steps",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=0,
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=eval_dataset)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
print("Saved adapter:", ADAPTER_DIR)

## Synthetic Validation Accuracy

This is the accuracy you can show in your notebook/report. Hidden test labels are unavailable, so final hidden accuracy/score is produced only by the evaluator's grading script.

In [ ]:
def extract_option(text):
    match = re.search(r"\b[1-5]\b", str(text))
    if match:
        return int(match.group(0))
    match = re.search(r"[1-5]", str(text))
    return int(match.group(0)) if match else 5


def predict_text_mcq(mcq_text):
    prompt = build_prompt(mcq_text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=4, do_sample=False, pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id)
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return extract_option(decoded), decoded


correct = 0
attempted = 0
for r in eval_records:
    mcq_text = format_mcq_text(r["question"], r["options"])
    pred, raw = predict_text_mcq(mcq_text)
    gold = int(r["correct_option"])
    correct += int(pred == gold)
    attempted += int(pred in {1, 2, 3, 4})

accuracy = correct / len(eval_records) if eval_records else 0
coverage = attempted / len(eval_records) if eval_records else 0
print(f"Synthetic validation accuracy: {accuracy:.3f} ({correct}/{len(eval_records)})")
print(f"Attempt coverage: {coverage:.3f} ({attempted}/{len(eval_records)})")

## OCR Test Images and Create Submission

In [ ]:
def preprocess_for_ocr(image_path):
    image = Image.open(image_path).convert("L")
    w, h = image.size
    if max(w, h) < 2200:
        image = image.resize((w * 2, h * 2))
    image = ImageEnhance.Contrast(image).enhance(1.8)
    return image.filter(ImageFilter.SHARPEN)


def ocr_image(image_path):
    if pytesseract is None:
        return ""
    try:
        return pytesseract.image_to_string(preprocess_for_ocr(image_path), config="--psm 6").strip()
    except Exception as exc:
        print("OCR failed:", image_path, exc)
        return ""


def read_test_rows(path):
    rows = []
    with Path(path).open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            image_name = row.get("image_name") or row.get("image_id") or row.get("id")
            if image_name:
                rows.append(str(image_name).strip())
    return rows


def find_image_path(image_name):
    raw = Path(image_name)
    candidates = [IMAGE_DIR / raw.name] if raw.suffix else [IMAGE_DIR / f"{image_name}.png", IMAGE_DIR / f"{image_name}.jpg", IMAGE_DIR / f"{image_name}.jpeg", IMAGE_DIR / image_name]
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


test_rows = read_test_rows(TEST_CSV)
submission = []
for i, image_name in enumerate(test_rows, start=1):
    image_path = find_image_path(image_name)
    text = ocr_image(image_path) if image_path.exists() else ""
    option, raw = predict_text_mcq(text) if text.strip() else (5, "")
    option = option if option in {1, 2, 3, 4, 5} else 5
    submission.append({"image_name": image_name, "option": option})
    print(f"{i}/{len(test_rows)} {image_name} -> {option} | raw={raw!r}")

with SUBMISSION_PATH.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image_name", "option"])
    writer.writeheader()
    writer.writerows(submission)

print("Wrote:", SUBMISSION_PATH)